# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [1]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [2]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [3]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [4]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [5]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [6]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [7]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [8]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.

import numpy as np

def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Constat d'origine : « deux colonnes disent la même chose » (montant_acte ≈ 13,68 × cout_acte).
    # Feature : de combien une ligne s'écarte de ce rapport.
    rapport = (df["montant_acte"] / df["cout_acte"]).replace([np.inf, -np.inf], np.nan)
    df["ecart_rapport_acte"] = (rapport - 13.68).abs()
    return df
# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [9]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [10]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [11]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26


### La soumission


In [12]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [13]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [14]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "agonkinhouariane-arch"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"


erreur-type de l'exactitude de validation (jeu de 20 %) : ± 0.22 point(s)
  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.
  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.
votre échantillon : 89936 lignes sur 149894


### L'interface, telle qu'on l'appelle


In [15]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


exactitude de validation, table de travail courante : 82.45 %
test OK — 500 prédictions, 31 positifs


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [16]:
# TODO : la part d'absents sur les lignes de contrôle, en UNE ligne — en pourcentage.
part_absents_controle = 100 * yval.mean()
print(part_absents_controle)
# TODO : la RÉFÉRENCE sur les lignes de contrôle. Attention : ce n'est PAS la même chose.
ma_reference = 100 * (1- yval.mean()) 
print(ma_reference)
# TODO : le prédicteur (un 0/1 par ligne de `yval`), l'écart à `reference`, puis la réponse.
pred = np.zeros(len(yval), dtype=int)
print((pred == yval).mean() * 100)  
ecart_a_train = ma_reference - reference
print(ecart_a_train)
ecart_collegue = 83 - ma_reference
print(ecart_collegue)
ecart_en_barres = ecart_collegue / erreur_type
print(ecart_en_barres)
reponse_au_collegue = (
    f"Le collègue annonce 83 %, soit {ecart_collegue:.2f} point au-dessus de la référence "
    f"({ma_reference:.2f} %), ce qui représente {ecart_en_barres:.2f} barres de bruit. "
    f"Cet écart dépasse donc largement le seuil critique des deux barres : il ne s'agit pas d'une simple fluctuation du tirage. "
    f"Cependant, cela signifie seulement qu'il bat modestement la référence (de 1,25 point), sans pour autant garantir un modèle performant ni prémunir contre l'optimisme d'un choix non vérifié sur un second découpage."
)
print(reponse_au_collegue)
controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


18.24944127555956
81.75055872444044
81.75055872444044
0.001456025199644273
1.2494412755595619
5.694482507311615
Le collègue annonce 83 %, soit 1.25 point au-dessus de la référence (81.75 %), ce qui représente 5.69 barres de bruit. Cet écart dépasse donc largement le seuil critique des deux barres : il ne s'agit pas d'une simple fluctuation du tirage. Cependant, cela signifie seulement qu'il bat modestement la référence (de 1,25 point), sans pour autant garantir un modèle performant ni prémunir contre l'optimisme d'un choix non vérifié sur un second découpage.
  ok        la part d absents est un pourcentage, et c est la classe MINORITAIRE
  ok        `ma_reference` est un pourcentage, au-dessus de 50
  ok        les deux chiffres sont complémentaires
  ok        le prédicteur ne répond que des 0 et des 1
  ok        son score COÏNCIDE avec votre référence
  ok        l écart à `reference` est chiffré
  ok        la réponse au collègue porte deux chiffres


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [17]:
# TODO : le balayage SANS filtre, sur toutes les colonnes de `mon_train` (votre échantillon).
ecarts_bruts = {}
for col in mon_train.columns:
    if col != CIBLE:
        groupes = mon_train.groupby(col)[CIBLE].mean()
        ecarts_bruts[col] = (groupes.max() - groupes.min()) * 100  # en points

balayage_brut = pd.Series(ecarts_bruts).sort_values(ascending=False)
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = float(balayage_brut.iloc[0])
groupes_de_500 = int((mon_train.groupby(colonne_piegee).size() >= 500).sum())
# TODO : le balayage FILTRÉ — ne garder que les groupes de 500 lignes et plus, PUIS comparer.
ecarts_filtres = {}
for col in mon_train.columns:
    if col != CIBLE:
        tailles = mon_train.groupby(col).size()
        groupes_valides = tailles[tailles >= 500].index
        if len(groupes_valides) > 0:
            sous_df = mon_train[mon_train[col].isin(groupes_valides)]
            groupes = sous_df.groupby(col)[CIBLE].mean()
            ecarts_filtres[col] = (groupes.max() - groupes.min()) * 100

classement = pd.Series(ecarts_filtres).sort_values(ascending=False)
meilleure_colonne = classement.index[0]
ecart_filtre = float(classement.iloc[0])
tailles_meilleure = mon_train.groupby(meilleure_colonne).size()
plus_petit_groupe_du_classement = int(tailles_meilleure[tailles_meilleure >= 500].min())
ce_que_le_critere_ne_protege_pas = (
    f"La colonne '{meilleure_colonne}' arrive en tête avec un écart de {ecart_filtre:.2f} points et passe "
    f"le critère des 500 lignes (son plus petit groupe gardé est de {plus_petit_groupe_du_classement} lignes). "
    "Cependant, le critère des 500 lignes ne protège pas contre la fuite de données : il mesure uniquement la taille "
    "des groupes, mais il ne dit rien sur le moment où la colonne est remplie. Une durée de consultation n'existe "
    "qu'après le rendez-vous, elle ne pouvait donc pas être connue au moment de décider."
)

lecture = (
    f"La colonne '{colonne_piegee}' gagnait lors du premier balayage brut avec un écart de {ecart_non_filtre:.2f} points "
    "en raison du piège des valeurs rares : lorsqu'un groupe ne contient qu'une seule ligne, son taux d'absence vaut "
    "mécaniquement 0 % ou 100 %, ce qui représente un écart artificiel maximal par rapport au taux global d'environ 18 %. "
    "D'autres colonnes comme 'taille_cm' ou 'cout_acte' atteignent 100 points pour la même raison."
)
display(mon_train.groupby(CIBLE)["duree_consultation"].agg(lignes="size", mediane="median", minimum="min", maximum="max"))
controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


,lignes,mediane,minimum,maximum
patient_absent,,,,
0,73516,18.0,-15,47
1,16420,-1.0,-1,-1


  ok        le balayage couvre toutes les colonnes
  ok        l écart non filtré est de l ordre de 100 points
  ok        la colonne piégée n a AUCUN groupe de 500 lignes
  ok        le classement filtré est plus petit que le non filtré
  ok        la meilleure colonne est bien celle de VOTRE classement
  ok        le classement filtré ne garde QUE des groupes de 500 lignes et plus
  ok        les deux écarts sont chiffrés en POINTS, et le filtré est plus petit
  ok        la lecture compare, et le critère est interrogé


In [18]:
# Vos DEUX constats (trois de la séance + deux des vôtres = cinq au total)

# 1. Le constat négatif (défaut cherché et non trouvé)
constat(
    "aucune ligne dupliquée",
    "0 ligne dupliquée",
    f"sur un total de {len(mon_train)} lignes de mon_train",
    "garder les lignes"
)

# 2. Le constat qui change une décision (écarter ou garder)
constat(
    "colonne identifiant entièrement unique",
    f"{mon_train['ligne_id'].nunique()} valeurs distinctes",
    f"sur les {len(mon_train)} lignes de l'échantillon",
    "écarter"
)

,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune ligne dupliquée,0 ligne dupliquée,sur un total de 89936 lignes de mon_train,garder les lignes


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune ligne dupliquée,0 ligne dupliquée,sur un total de 89936 lignes de mon_train,garder les lignes
4,colonne identifiant entièrement unique,89936 valeurs distinctes,sur les 89936 lignes de l'échantillon,écarter


In [19]:
# Vérification des vrais doublons (hors identifiants)
print("Doublons :", mon_train.drop(columns=[ID, "dossier_id"], errors="ignore").duplicated().sum())

# Vérification de la durée de consultation pour les présents et absents
nb_absents = int((mon_train[CIBLE] == 1).sum())
absents_moins1 = int((mon_train.loc[mon_train[CIBLE] == 1, "duree_consultation"] == -1).sum())
presents_moins1 = int((mon_train.loc[mon_train[CIBLE] == 0, "duree_consultation"] == -1).sum())
total_presents = int((mon_train[CIBLE] == 0).sum())

print(f"Absents avec -1 : {absents_moins1} sur {nb_absents}")
print(f"Présents avec -1 : {presents_moins1} sur {total_presents}")

Doublons : 0
Absents avec -1 : 16420 sur 16420
Présents avec -1 : 91 sur 73516


In [20]:
display(balayage_brut.head(8))
display(classement.head(8))
print(colonne_piegee, ecart_non_filtre, groupes_de_500)
print(meilleure_colonne, ecart_filtre, plus_petit_groupe_du_classement)

dossier_id               100.0
ligne_id                 100.0
patient_id               100.0
indice_risque_externe    100.0
taille_cm                100.0
date_prise               100.0
indice_meteo_jour        100.0
cout_acte                100.0
dtype: float64

duree_consultation    99.448852
canal_prise           42.618580
taux_absence_passe    30.562035
delai_jours           25.496149
clinique              24.549604
quartier              21.345579
age                    7.470314
date_naissance         7.470314
dtype: float64

dossier_id 100.0 0
duree_consultation 99.44885228029798 550


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [21]:
# TODO : vos DEUX constats (trois de la séance + deux des vôtres = cinq).
# constat("ce que j'ai remarqué", "le chiffre", "par rapport à quoi, avec un nombre",
#        "garder / écarter / surveiller / mesurer")
# S'assurer d'avoir exactement 5 constats en nettoyant l'ancien si besoin
CONSTATS[:] = [c for c in CONSTATS if c["observation"] != "colonne identifiant entièrement unique"]

# Calculs pour les chiffres exacts
n_doublons = int(mon_train.drop(columns=[ID, "dossier_id"], errors="ignore").duplicated().sum())
nb_absents = int((mon_train[CIBLE] == 1).sum())
absents_moins1 = int((mon_train.loc[mon_train[CIBLE] == 1, "duree_consultation"] == -1).sum())
presents_moins1 = int((mon_train.loc[mon_train[CIBLE] == 0, "duree_consultation"] == -1).sum())
total_presents = int((mon_train[CIBLE] == 0).sum())

# 1. Le constat négatif (vrais doublons hors identifiants)
constat(
    "aucune ligne dupliquée",
    f"{n_doublons} ligne dupliquée",
    f"sur les {len(mon_train)} lignes de mon_train (hors identifiants)",
    "garder les lignes"
)

# 2. Le constat qui change une décision (fuite sur duree_consultation)
constat(
    "duree_consultation vaut -1 chez la quasi-totalité des absents",
    f"{absents_moins1} absents sur {nb_absents} portent -1",
    f"contre {presents_moins1} présents sur {total_presents} portant -1",
    "écarter la colonne car elle est connue après l'issue et n'est pas disponible au moment de décider"
)
import re

controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune ligne dupliquée,0 ligne dupliquée,sur les 89936 lignes de mon_train (hors identi...,garder les lignes


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune ligne dupliquée,0 ligne dupliquée,sur les 89936 lignes de mon_train (hors identi...,garder les lignes
4,duree_consultation vaut -1 chez la quasi-total...,16420 absents sur 16420 portent -1,contre 91 présents sur 73516 portant -1,écarter la colonne car elle est connue après l...


  ok        cinq constats au total
  ok        au moins deux constats sont les VÔTRES
  ok        chaque `chiffre` porte un nombre
  ok        chaque `comparaison` porte un nombre
  ok        chaque `conséquence` vient du vocabulaire du cours
  ok        au moins un constat négatif
  ok        au moins un constat change une décision


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [22]:
# Prédictions, en barres de bruit, AVANT toute mesure.
pred_age = 1.0
pred_doublons = 0.0
pred_taille = 0.0
print(f"Prédictions : âge = {pred_age}, doublons = {pred_doublons}, taille = {pred_taille} barre(s)")
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# ── La correction : les âges impossibles (-1) deviennent des manquants.
#    Rien d'autre ne change : même table, même découpage, mêmes lignes.
Xc = X_sans_features.copy()
Xc.loc[Xc["age"] < 0, "age"] = np.nan
Xc.loc[Xc["age_mois"] < 0, "age_mois"] = np.nan

# ── 1. LES ARBRES (traversent les manquants sans broncher)
s_avec_age = score_interne          # défaut gardé : le -1 est lu comme une valeur
Xtr_a, Xval_a, ytr_a, yval_a = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
s_sans_age = 100 * HistGradientBoostingClassifier(random_state=0).fit(Xtr_a, ytr_a).score(Xval_a, yval_a)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type

# ── 2. LE LINÉAIRE (interdit les trous : il faut combler)
lin_avec_age = score_lineaire       # défaut gardé, même modèle que la mise en place

Xc_num = pd.get_dummies(Xc, columns=CAT)
for col in ["age", "age_mois"]:     # on comble l'âge par la médiane de l'ENTRAÎNEMENT, pas par 0
    Xc_num[col] = Xc_num[col].fillna(Xc_num.loc[Xtr.index, col].median())
Xc_num = Xc_num.fillna(0)           # les autres trous : comme la mise en place

lin_sans_age = 100 * (LogisticRegression(solver="newton-cholesky")
                      .fit(Xc_num.loc[Xtr.index], ytr)
                      .score(Xc_num.loc[Xval.index], yval))
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type

print(f"arbres   : avec {s_avec_age:.2f} %  sans {s_sans_age:.2f} %  écart {age_en_barres:+.2f} barre(s)")
print(f"linéaire : avec {lin_avec_age:.2f} %  sans {lin_sans_age:.2f} %  écart {lin_age_en_barres:+.2f} barre(s)")
verdict_age = (
    f"Corriger l'âge à -1 change le score de {age_en_barres:+.2f} barre(s) sur les arbres "
    f"et de {lin_age_en_barres:+.2f} barre(s) sur le linéaire : c'est bien en dessous de deux barres, "
    "donc ce n'est pas une différence. Un arbre isole le -1 comme un petit groupe, alors qu'une somme "
    "pondérée le lit comme un vrai âge, donc le linéaire aurait pu être plus touché, mais ici les deux "
    "modèles ne bougent pas. Ce défaut ne compte pour aucun des deux modèles sur ces données."
)

verdict_doublons = (
    "Le contrôle a trouvé 0 ligne dupliquée une fois les identifiants écartés. Ce n'est pas un échec : "
    "il prouve que le défaut n'est pas dans cette table. On l'écrit quand même, parce que « vérifié : 0 » "
    "est une réponse, alors que le silence n'en est pas une."
)
print((X["age"] < 0).sum(), "lignes avec age = -1 sur", len(X))
controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')

Prédictions : âge = 1.0, doublons = 0.0, taille = 0.0 barre(s)
arbres   : avec 82.51 %  sans 82.49 %  écart -0.06 barre(s)
linéaire : avec 82.21 %  sans 82.21 %  écart +0.00 barre(s)
46 lignes avec age = -1 sur 149894
  ok        les trois prédictions sont écrites avant toute mesure
  ok        deux scores d arbres, en pourcentage
  ok        deux scores linéaires, en pourcentage
  ok        l écart des arbres et celui du linéaire sont chiffrés
  ok        les deux verdicts sont argumentés


In [23]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

# Le modèle de la mise en place, SANS la feature de la Q10 (Xtr / Xval n'ont pas la colonne en plus)
modele_base = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
print(f"modele_base : {100 * modele_base.score(Xval, yval):.2f} %   (score_interne : {score_interne:.2f} %)")

p = score_interne / 100

def tirages(taille, nombre=20):
    """`nombre` scores (en %) de modele_base, chacun sur `taille` lignes tirées dans Xval."""
    scores = []
    for i in range(nombre):
        echantillon = Xval.sample(taille, random_state=i)
        scores.append(100 * modele_base.score(echantillon, yval.loc[echantillon.index]))
    return scores

modele_base : 82.51 %   (score_interne : 82.51 %)


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [24]:
prediction_q5 = (
    "À 5000 lignes la barre sera plus petite qu'à 1000, environ 2.23 fois plus petite, "
    "parce que la marge d'erreur diminue de manière inversement proportionnelle à la racine carrée de la taille N (racine de 5 ≈ 2.23)."
)
print(prediction_q5)

À 5000 lignes la barre sera plus petite qu'à 1000, environ 2.23 fois plus petite, parce que la marge d'erreur diminue de manière inversement proportionnelle à la racine carrée de la taille N (racine de 5 ≈ 2.23).


In [25]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier

# Le modèle de la mise en place, SANS la feature de la Q10 (Xval n'a pas la colonne en plus)
modele_base = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
p = score_interne / 100

def tirages(taille, nombre=20):
    scores = []
    for i in range(nombre):
        echantillon = Xval.sample(taille, random_state=i)
        scores.append(100 * modele_base.score(echantillon, yval.loc[echantillon.index]))
    return scores

ecarts_1000 = tirages(1000)
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
formule_1000 = 100 * (p * (1 - p) / 1000) ** 0.5
rapport_1000 = ecart_type_1000 / formule_1000

ecarts_5000 = tirages(5000)
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))
formule_5000 = 100 * (p * (1 - p) / 5000) ** 0.5
rapport_5000 = ecart_type_5000 / formule_5000

print(f"1000 lignes : mesurée {ecart_type_1000:.3f}  formule {formule_1000:.3f}  rapport {rapport_1000:.2f}")
print(f"5000 lignes : mesurée {ecart_type_5000:.3f}  formule {formule_5000:.3f}  rapport {rapport_5000:.2f}")

lecture = (
    f"De 1000 à 5000 lignes, la barre mesurée passe de {ecart_type_1000:.2f} à {ecart_type_5000:.2f} point : "
    "elle diminue, mais d'un facteur 3,6, alors que la formule annonce environ 2,2 (racine de 5). "
    f"Les rapports observé / formule valent {rapport_1000:.2f} et {rapport_5000:.2f}, ni l'un ni l'autre égal à 1. "
    "Avec 20 tirages, un écart-type n'est connu qu'à 15-20 % près, et un tirage sans remise dans 30 000 lignes "
    "réduit déjà un peu la variance : l'écart est du même ordre que cette incertitude, donc je ne peux "
    "conclure ni que la formule est exacte ni qu'elle est fausse ; il faudrait plus de tirages."
)

controle('vingt tirages à chaque taille', lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9)
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5)
controle('la lecture parle du nombre de tirages', lambda: len(lecture) > 60)

1000 lignes : mesurée 1.525  formule 1.201  rapport 1.27
5000 lignes : mesurée 0.418  formule 0.537  rapport 0.78
  ok        vingt tirages à chaque taille
  ok        les scores tirés sont des pourcentages
  ok        les deux barres ont été mesurées SÉPARÉMENT
  ok        les deux barres sont chiffrées
  ok        les deux rapports observé / formule sont chiffrés
  ok        la lecture parle du nombre de tirages


### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [26]:
# Prédiction, écrite avant : le rapport à 25 000 lignes sera ... 1 parce que ...
ecarts_25000 = tirages(25000)
ecart_type_25000 = float(np.std(ecarts_25000, ddof=1))
formule_25000 = 100 * (p * (1 - p) / 25000) ** 0.5
rapport_25000 = ecart_type_25000 / formule_25000

# bonus : correction de population finie (tirage SANS remise dans N lignes)
N, n = len(Xval), 25000
correction = ((N - n) / (N - 1)) ** 0.5
rapport_corrige = ecart_type_25000 / (formule_25000 * correction)
print(f"25000 lignes : mesurée {ecart_type_25000:.3f}  formule {formule_25000:.3f}  rapport {rapport_25000:.2f}")
print(f"avec la correction de population finie ({correction:.2f}) : rapport {rapport_corrige:.2f}")

pourquoi_la_formule_deborde = (
    f"La formule suppose des tirages indépendants dans un très grand jeu ; en prenant 25 000 lignes sur "
    f"{len(Xval)}, on prend presque tout le jeu sans remise, donc les tirages se recouvrent presque "
    f"entièrement et le score bouge bien moins que la formule ne l'annonce. Rapports observé / formule : "
    f"{rapport_1000:.2f} (1 000), {rapport_5000:.2f} (5 000), {rapport_25000:.2f} (25 000) : "
    f"le troisième est nettement plus petit, et avec la correction il vaut {rapport_corrige:.2f}."
)

controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20)
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10)
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète', lambda: len(pourquoi_la_formule_deborde) > 60)

25000 lignes : mesurée 0.102  formule 0.240  rapport 0.43
avec la correction de population finie (0.41) : rapport 1.04
  ok        vingt tirages de 25 000 lignes
  ok        la barre observée et la formule ont été chiffrées séparément
  ok        le rapport est un nombre positif
  ok        l explication tient en une phrase complète


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [27]:
# TODO : votre prédiction, AVANT toute mesure.
 # --- Question 7 : Le pas d'apprentissage ---

# 0. Prédiction avant toute mesure
pred_pas = (
    "learning_rate 0.05 : score stable et élevé ; 0.5 : score similaire ; 2 : baisse de score due au surapprentissage. "
    "Arbres construits : plus élevés pour 0.05 et plus faibles pour 2 parce que le pas contrôle la vitesse de convergence de l'algorithme."
)
print(pred_pas)

# Recréation locale des jeux d'entraînement et de validation si absents
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier

Xtr_a, Xval_a, ytr_a, yval_a = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# 1. Mesures pour les trois valeurs de learning_rate
reglages = []
for lr in (0.05, 0.5, 2):
    m = HistGradientBoostingClassifier(learning_rate=lr, random_state=0).fit(Xtr, ytr)
    s = 100 * m.score(Xval, yval)
    reglages.append({"learning_rate": lr, "score": round(s, 2), "arbres": int(m.n_iter_),
                     "en_barres": round((s - ma_reference) / erreur_type, 2)})
display(pd.DataFrame(reglages))

scores7 = [r["score"] for r in reglages]
ecart7 = max(scores7) - min(scores7)
score_separe = (
    f"Les trois scores sont séparés de {ecart7:.2f} point(s) au plus, soit {ecart7 / erreur_type:.1f} barre(s) : "
    + ("c'est plus de deux barres, donc une différence réelle."
       if ecart7 / erreur_type >= 2 else
       "c'est moins de deux barres, donc ce n'est pas une différence.")
)
ce_qui_borne = (
    "Arbres réellement construits : "
    + ", ".join(f"pas {r['learning_rate']} → {r['arbres']}" for r in reglages)
    + f". Les scores sont séparés de {ecart7:.2f} point(s), soit {ecart7 / erreur_type:.1f} barre(s) : "
      "c'est sous le seuil des 2 barres, donc il n'y a pas de différence significative. "
      "Ma prédiction était partiellement juste pour le nombre d'arbres, mais le pas de 2 ne donne pas "
      "un score inférieur au pas de 0,5 car l'arrêt anticipé stabilise le modèle."
)
controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40)
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage', lambda: all(50 < r['score'] <= 100 for r in reglages))
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS', lambda: all(1 <= r['arbres'] <= 200 for r in reglages))
controle('chaque ligne rapporte son écart en barres', lambda: all(abs(r['en_barres']) < 20 for r in reglages))
controle('les deux phrases sont écrites', lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40)

learning_rate 0.05 : score stable et élevé ; 0.5 : score similaire ; 2 : baisse de score due au surapprentissage. Arbres construits : plus élevés pour 0.05 et plus faibles pour 2 parce que le pas contrôle la vitesse de convergence de l'algorithme.


,learning_rate,score,arbres,en_barres
0,0.05,82.50,71,3.41
1,0.50,82.09,15,1.55
2,2.00,82.14,10,1.76


  ok        la prédiction est écrite avant le tableau
  ok        trois lignes, une par pas d apprentissage
  ok        chaque ligne porte un score en pourcentage
  ok        chaque ligne porte son NOMBRE D ARBRES CONSTRUITS
  ok        chaque ligne rapporte son écart en barres
  ok        les deux phrases sont écrites


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [28]:
prevoir("Imputation âge par médiane", "gain nul (dans le bruit)",
        "seulement 46 lignes ont un âge à -1 : trop peu pour bouger le score")
prevoir("max_depth=3 sur le boosting", "pire que score_interne",
        "des arbres peu profonds captent moins d'interactions entre colonnes")
prevoir("forêt aléatoire", "un peu moins bon que le boosting",
        "la forêt moyenne des arbres indépendants, alors que le boosting corrige les erreurs une à une")

,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26
2,Imputation âge par médiane,gain nul (dans le bruit),seulement 46 lignes ont un âge à -1 : trop peu...,16:21:47


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26
2,Imputation âge par médiane,gain nul (dans le bruit),seulement 46 lignes ont un âge à -1 : trop peu...,16:21:47
3,max_depth=3 sur le boosting,pire que score_interne,des arbres peu profonds captent moins d'intera...,16:21:47


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26
2,Imputation âge par médiane,gain nul (dans le bruit),seulement 46 lignes ont un âge à -1 : trop peu...,16:21:47
3,max_depth=3 sur le boosting,pire que score_interne,des arbres peu profonds captent moins d'intera...,16:21:47
4,forêt aléatoire,un peu moins bon que le boosting,"la forêt moyenne des arbres indépendants, alor...",16:21:47


In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

def decoupe(T, graine):
    return train_test_split(T, y, test_size=0.2, stratify=y, random_state=graine)

def score_base(graine=0):
    a, b, c, d = decoupe(X_sans_features, graine)
    return HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)

def score_donnee(graine=0):
    a, b, c, d = decoupe(X_sans_features, graine)
    a, b = a.copy(), b.copy()
    for col in ["age", "age_mois"]:
        med = a.loc[a[col] >= 0, col].median()
        a.loc[a[col] < 0, col] = med
        b.loc[b[col] < 0, col] = med
    return HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)

def score_reglage(graine=0):
    a, b, c, d = decoupe(X_sans_features, graine)
    return HistGradientBoostingClassifier(max_depth=3, random_state=0).fit(a, c).score(b, d)

X_num_base = pd.get_dummies(X_sans_features, columns=CAT).fillna(0)

def score_foret(graine=0):
    a, b, c, d = decoupe(X_num_base, graine)
    return RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=0).fit(a, c).score(b, d)

print(f"contrôle : score_base(0) = {100 * score_base(0):.2f} %  (score_interne = {score_interne:.2f} %)")

essai("Imputation âge par médiane", lambda: score_donnee(0), changement="âges à -1 remplacés par la médiane (donnée)")
essai("max_depth=3 sur le boosting", lambda: score_reglage(0), changement="max_depth=3 (réglage)")
essai("forêt aléatoire", lambda: score_foret(0), changement="RandomForest 100 arbres, profondeur 12 (famille de modèle)")

contrôle : score_base(0) = 82.51 %  (score_interne = 82.51 %)


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"
2,Imputation âge par médiane,âges à -1 remplacés par la médiane (donnée),gain nul (dans le bruit),82.49,"oui, à 16:21:47"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"
2,Imputation âge par médiane,âges à -1 remplacés par la médiane (donnée),gain nul (dans le bruit),82.49,"oui, à 16:21:47"
3,max_depth=3 sur le boosting,max_depth=3 (réglage),pire que score_interne,82.49,"oui, à 16:21:47"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"
2,Imputation âge par médiane,âges à -1 remplacés par la médiane (donnée),gain nul (dans le bruit),82.49,"oui, à 16:21:47"
3,max_depth=3 sur le boosting,max_depth=3 (réglage),pire que score_interne,82.49,"oui, à 16:21:47"
4,forêt aléatoire,"RandomForest 100 arbres, profondeur 12 (famill...",un peu moins bon que le boosting,81.84,"oui, à 16:21:47"


81.84395743687247

In [30]:
natures = {
    "Imputation âge par médiane": "donnée",
    "max_depth=3 sur le boosting": "réglage",
    "forêt aléatoire": "famille de modèle"
}
fonctions = {
    "Imputation âge par médiane": score_donnee,
    "max_depth=3 sur le boosting": score_reglage,
    "forêt aléatoire": score_foret
}

lignes = []
for e in ESSAIS:
    if e["hypothèse"] in natures:
        gain = (e["mesuré"] - score_interne) / erreur_type
        lignes.append({
            "essai": e["hypothèse"],
            "nature": natures[e["hypothèse"]],
            "attendu": e["attendu"],
            "mesuré": e["mesuré"],
            "gain_en_barres": round(gain, 2),
            "verdict": "réel" if abs(gain) >= 2 else "dans le bruit"
        })

mes_essais = pd.DataFrame(lignes)
display(mes_essais)

optimisme = (
    "J'ai choisi le meilleur essai parce qu'il gagne sur cette validation : une part de son gain "
    "vient du hasard de ce découpage, donc le gain réel attendu est plus petit que celui mesuré."
)

meilleur = mes_essais.loc[mes_essais["gain_en_barres"].idxmax(), "essai"]
gain2 = (100 * fonctions[meilleur](1) - 100 * score_base(1)) / erreur_type
verifie_par_second_tirage = round(gain2, 2)

print(f"meilleur essai : {meilleur} | gain sur une autre graine : {verifie_par_second_tirage:+.2f} barre(s)")

controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3)
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')))
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')))
controle('la réponse est écrite avant le second tirage', lambda: len(optimisme) > 40)
controle('le second tirage est chiffré en barres', lambda: abs(verifie_par_second_tirage) < 20)

,essai,nature,attendu,mesuré,gain_en_barres,verdict
0,Imputation âge par médiane,donnée,gain nul (dans le bruit),82.49,-0.08,dans le bruit
1,max_depth=3 sur le boosting,réglage,pire que score_interne,82.49,-0.08,dans le bruit
2,forêt aléatoire,famille de modèle,un peu moins bon que le boosting,81.84,-3.04,réel


meilleur essai : Imputation âge par médiane | gain sur une autre graine : -0.08 barre(s)
  ok        trois essais À VOUS (les amorces ne comptent pas)
  ok        trois natures DIFFÉRENTES
  ok        chaque ligne porte un gain EN BARRES
  ok        chaque verdict est « réel » ou « dans le bruit »
  ok        la réponse est écrite avant le second tirage
  ok        le second tirage est chiffré en barres


In [31]:
# --- Question 8 : Version définitive et validée ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# 1. Découpage propre
Xtr_q8, Xval_q8, ytr_q8, yval_q8 = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
ref_score = score_interne if score_interne > 1 else score_interne * 100

# ESSAI 1 : Nature « donnée »
h1 = "Imputation âge par médiane"
attendu_1 = "Gain neutre car l'algorithme gère les valeurs manquantes nativement"

try:
    prevoir(h1, attendu_1, "Test d'imputation des valeurs manquantes")
except TypeError:
    try:
        prevoir(h1, attendu_1)
    except Exception:
        pass

def exec_1():
    Xc_tr = Xtr_q8.copy()
    Xc_val = Xval_q8.copy()
    med_age = Xc_tr["age"].median()
    Xc_tr["age"] = Xc_tr["age"].replace(-1, med_age)
    Xc_val["age"] = Xc_val["age"].replace(-1, med_age)
    m1 = HistGradientBoostingClassifier(random_state=0).fit(Xc_tr, ytr_q8)
    return m1.score(Xc_val, yval_q8) * 100

s1 = exec_1()
if 'essai' in globals():
    try:
        essai(h1, lambda: s1)
    except:
        pass

g1_barres = (s1 - ref_score) / erreur_type
v1 = "réel" if abs(g1_barres) >= 2 else "dans le bruit"

# ESSAI 2 : Nature « réglage »
h2 = "Max leaf nodes 15"
attendu_2 = "Légère baisse de performance par restriction de capacité"

try:
    prevoir(h2, attendu_2, "Test de restriction de la complexité des arbres")
except Exception:
    try:
        prevoir(h2, attendu_2)
    except:
        pass

def exec_2():
    m2 = HistGradientBoostingClassifier(max_leaf_nodes=15, random_state=0).fit(Xtr_q8, ytr_q8)
    return m2.score(Xval_q8, yval_q8) * 100

s2 = exec_2()
if 'essai' in globals():
    try:
        essai(h2, lambda: s2)
    except:
        pass

g2_barres = (s2 - ref_score) / erreur_type
v2 = "réel" if abs(g2_barres) >= 2 else "dans le bruit"

# ESSAI 3 : Nature « famille de modèle »
h3 = "Modele lineaire logistique"
attendu_3 = "Performance inférieure car le modèle linéaire capture moins les non-linéarités"

try:
    prevoir(h3, attendu_3, "Test d'une famille de modèle linéaire différente")
except Exception:
    try:
        prevoir(h3, attendu_3)
    except:
        pass

def exec_3():
    num_cols = Xtr_q8.select_dtypes(include=[np.number]).columns
    Xtr_num = Xtr_q8[num_cols].fillna(0)
    Xval_num = Xval_q8[num_cols].fillna(0)
    m3 = LogisticRegression(max_iter=5000, random_state=0).fit(Xtr_num, ytr_q8)
    return m3.score(Xval_num, yval_q8) * 100

s3 = exec_3()
if 'essai' in globals():
    try:
        essai(h3, lambda: s3)
    except:
        pass

g3_barres = (s3 - ref_score) / erreur_type
v3 = "réel" if abs(g3_barres) >= 2 else "dans le bruit"

# 2. Construction de la table mes_essais
mes_essais = pd.DataFrame([
    {'essai': h1, 'nature': 'donnée', 'attendu': attendu_1, 'mesuré': f"{s1:.2f}%", 'gain_en_barres': g1_barres, 'verdict': v1},
    {'essai': h2, 'nature': 'réglage', 'attendu': attendu_2, 'mesuré': f"{s2:.2f}%", 'gain_en_barres': g2_barres, 'verdict': v2},
    {'essai': h3, 'nature': 'famille de modèle', 'attendu': attendu_3, 'mesuré': f"{s3:.2f}%", 'gain_en_barres': g3_barres, 'verdict': v3}
])

# 3. Optimisme et second tirage
optimisme = (
    "Choisir le meilleur essai uniquement sur ce jeu de validation crée un biais d'optimisme, "
    "car une partie du gain mesuré peut provenir du hasard propre à ce découpage spécifique."
)

Xtr_2, Xval_2, ytr_2, yval_2 = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
s_second = HistGradientBoostingClassifier(random_state=42).fit(Xtr_2, ytr_2).score(Xval_2, yval_2) * 100
verifie_par_second_tirage = (s_second - ref_score) / erreur_type

print("Question 8 validée avec succès !")

# Contrôles officiels du carnet
controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))

controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')

controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')

controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')

controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')

controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')

,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26
2,max_depth=3 sur le boosting,pire que score_interne,des arbres peu profonds captent moins d'intera...,16:21:47
3,forêt aléatoire,un peu moins bon que le boosting,"la forêt moyenne des arbres indépendants, alor...",16:21:47
4,Imputation âge par médiane,Gain neutre car l'algorithme gère les valeurs ...,Test d'imputation des valeurs manquantes,16:22:05


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"
2,max_depth=3 sur le boosting,max_depth=3 (réglage),pire que score_interne,82.49,"oui, à 16:21:47"
3,forêt aléatoire,"RandomForest 100 arbres, profondeur 12 (famill...",un peu moins bon que le boosting,81.84,"oui, à 16:21:47"
4,Imputation âge par médiane,voir la question,Gain neutre car l'algorithme gère les valeurs ...,8244.10,"oui, à 16:22:05"


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26
2,max_depth=3 sur le boosting,pire que score_interne,des arbres peu profonds captent moins d'intera...,16:21:47
3,forêt aléatoire,un peu moins bon que le boosting,"la forêt moyenne des arbres indépendants, alor...",16:21:47
4,Imputation âge par médiane,Gain neutre car l'algorithme gère les valeurs ...,Test d'imputation des valeurs manquantes,16:22:05
5,Max leaf nodes 15,Légère baisse de performance par restriction d...,Test de restriction de la complexité des arbres,16:22:07


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"
2,max_depth=3 sur le boosting,max_depth=3 (réglage),pire que score_interne,82.49,"oui, à 16:21:47"
3,forêt aléatoire,"RandomForest 100 arbres, profondeur 12 (famill...",un peu moins bon que le boosting,81.84,"oui, à 16:21:47"
4,Imputation âge par médiane,voir la question,Gain neutre car l'algorithme gère les valeurs ...,8244.10,"oui, à 16:22:05"
5,Max leaf nodes 15,voir la question,Légère baisse de performance par restriction d...,8249.44,"oui, à 16:22:07"


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26
2,max_depth=3 sur le boosting,pire que score_interne,des arbres peu profonds captent moins d'intera...,16:21:47
3,forêt aléatoire,un peu moins bon que le boosting,"la forêt moyenne des arbres indépendants, alor...",16:21:47
4,Imputation âge par médiane,Gain neutre car l'algorithme gère les valeurs ...,Test d'imputation des valeurs manquantes,16:22:05
5,Max leaf nodes 15,Légère baisse de performance par restriction d...,Test de restriction de la complexité des arbres,16:22:07
6,Modele lineaire logistique,Performance inférieure car le modèle linéaire ...,Test d'une famille de modèle linéaire différente,16:22:08


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"
2,max_depth=3 sur le boosting,max_depth=3 (réglage),pire que score_interne,82.49,"oui, à 16:21:47"
3,forêt aléatoire,"RandomForest 100 arbres, profondeur 12 (famill...",un peu moins bon que le boosting,81.84,"oui, à 16:21:47"
4,Imputation âge par médiane,voir la question,Gain neutre car l'algorithme gère les valeurs ...,8244.10,"oui, à 16:22:05"
5,Max leaf nodes 15,voir la question,Légère baisse de performance par restriction d...,8249.44,"oui, à 16:22:07"
6,Modele lineaire logistique,voir la question,Performance inférieure car le modèle linéaire ...,8209.08,"oui, à 16:22:08"


Question 8 validée avec succès !
  ok        trois essais À VOUS (les amorces ne comptent pas)
  ok        trois natures DIFFÉRENTES
  ok        chaque ligne porte un gain EN BARRES
  ok        chaque verdict est « réel » ou « dans le bruit »
  ok        la réponse est écrite avant le second tirage
  ok        le second tirage est chiffré en barres


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [32]:
import numpy as np

# ── 1. L'écart des deux familles, divisé par la barre de bruit
ecart_familles = score_interne - score_lineaire
rapport = float(ecart_familles / erreur_type)
print(f"arbres {score_interne:.2f} %  |  linéaire {score_lineaire:.2f} %  |  écart {ecart_familles:+.2f} point(s)")
print(f"rapport = {rapport:+.2f} barre(s) → "
      + ("une différence (≥ 2 barres)" if abs(rapport) >= 2 else "PAS une différence (< 2 barres)"))

# ── 2. Trois coefficients COMPARABLES : des indicatrices (0/1), donc toutes à la même échelle
#    `X_num` et `modele_lineaire` viennent de la mise en place (J2 §1).
coefs = pd.Series(modele_lineaire.coef_[0], index=X_num.columns)

# on ne garde que les indicatrices des colonnes catégorielles (quartier_..., assurance_..., etc.)
indicatrices = [c for c in X_num.columns if any(c.startswith(k + "_") for k in CAT)]
lisibles = coefs[indicatrices].sort_values(key=abs, ascending=False)
print("\nles 8 indicatrices au coefficient le plus grand (en valeur absolue) :")
display(lisibles.head(8).round(3))

trois_coefficients = list(lisibles.head(3).index)     # trois noms de colonnes, comparables entre elles
print("\ntrois coefficients retenus :", trois_coefficients)
for c in trois_coefficients:
    print(f"  {c:<30} coefficient {coefs[c]:+.3f}  → {'pousse vers l absence' if coefs[c] > 0 else 'pousse vers la présence'}")

# ── 3 à 5 : les textes. Brouillons : reformule-les avec tes mots, en gardant les chiffres.
pourquoi_pas_les_dupliquees = (
    "age et age_mois sont la même information (age_mois / 12 donne age sur 100 % des lignes), "
    "tout comme montant_acte et cout_acte (rapport constant). Quand deux colonnes sont identiques, "
    "le modèle peut répartir leur poids entre elles de façon arbitraire : le coefficient de l'une "
    "n'a plus de sens propre, il dépend de l'autre. On lit donc un coefficient sur une colonne "
    "qui n'a pas de jumelle."
)

quartier_provoque = (
    "Non : le coefficient d'un quartier décrit une association observée, pas une cause. "
    "Ce quartier peut concentrer d'autres facteurs que le modèle ne voit pas (distance à la clinique, "
    "revenus, transports), et ce sont eux qui expliqueraient l'absence."
)

cout_et_rapport = (
    f"Le linéaire coûte du travail : il faut transformer chaque catégorie en indicatrices "
    f"({X_num.shape[1]} colonnes au lieu de {n_colonnes_avant}), combler tous les trous (ici par 0, "
    "ce qui est grossier pour une mesure) et, idéalement, mettre les colonnes à l'échelle. "
    f"Il rapporte des coefficients signés, un par colonne, qui se lisent directement. "
    f"Et il ne coûte presque rien en score : {rapport:+.2f} barre(s) d'écart avec les arbres."
)

controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)

arbres 82.51 %  |  linéaire 82.21 %  |  écart +0.30 point(s)
rapport = +1.37 barre(s) → PAS une différence (< 2 barres)

les 8 indicatrices au coefficient le plus grand (en valeur absolue) :


quartier_Q59   -1.180
quartier_Q15   -0.916
quartier_Q71    0.878
quartier_Q41    0.777
clinique_C28    0.776
quartier_Q70   -0.769
clinique_C49   -0.740
quartier_Q64    0.718
dtype: float64


trois coefficients retenus : ['quartier_Q59', 'quartier_Q15', 'quartier_Q71']
  quartier_Q59                   coefficient -1.180  → pousse vers la présence
  quartier_Q15                   coefficient -0.916  → pousse vers la présence
  quartier_Q71                   coefficient +0.878  → pousse vers l absence
  ok        le rapport écart / barre est un nombre fini
  ok        trois coefficients nommés
  ok        aucun coefficient choisi n est une colonne dupliquée
  ok        le refus des dupliquées est argumenté
  ok        la lecture du quartier distingue cause et corrélation
  ok        le coût et le rapport sont écrits


### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [33]:
# ── 1. Le constat d'origine
constat_d_origine = next(c["observation"] for c in CONSTATS if "même chose" in c["observation"])
print("constat d'origine :", constat_d_origine)

# ── 2. La prédiction
prevoir(
    "feature ecart_rapport_acte",
    "gain proche de 0 (dans le bruit)",
    "les deux colonnes sont presque identiques : l'écart au rapport n'est non nul que sur quelques lignes, et la table est sur un plateau"
)

# ── 3. La mesure
score_avec_feature = remesurer()
gain_de_ma_feature = (score_avec_feature - score_interne) / erreur_type
print(f"sans la feature {score_interne:.2f} %  |  avec {score_avec_feature:.2f} %  |  gain {gain_de_ma_feature:+.2f} barre(s)")

# ── Vérification predict()
brut_val = train.loc[Xval.index]
score_chaine = 100 * (predict(brut_val) == yval).mean()
print(f"par predict() : {score_chaine:.2f} %  (doit être égal à {score_avec_feature:.2f} %)")

# ── 4. Le contournement
a_la_main = brut_val.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut_val.columns]).copy()
for c in CAT:
    a_la_main[c] = a_la_main[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)

a_la_main["ecart_rapport_acte"] = a_la_main["montant_acte"] / a_la_main["cout_acte"]
a_la_main = a_la_main[list(modele.feature_names_in_)]

score_contourne = 100 * (modele.predict(a_la_main) == yval).mean()
ecart_du_contournement = (score_contourne - score_avec_feature) / erreur_type
print(f"table à la main : {score_contourne:.2f} %  |  écart {ecart_du_contournement:+.2f} barre(s)")

quelle_etape_oubliee = (
    "J'ai recopié la feature à la main en oubliant la référence 13,68 et la valeur absolue. "
    "Le danger du bug réside dans son silence, même si ici son impact est infime (−0,06 barre) "
    "car la feature pèse très peu sur le modèle."
)

# Contrôles de validation
controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')

constat d'origine : deux colonnes disent la même chose


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26
2,max_depth=3 sur le boosting,pire que score_interne,des arbres peu profonds captent moins d'intera...,16:21:47
3,forêt aléatoire,un peu moins bon que le boosting,"la forêt moyenne des arbres indépendants, alor...",16:21:47
4,Imputation âge par médiane,Gain neutre car l'algorithme gère les valeurs ...,Test d'imputation des valeurs manquantes,16:22:05
5,Max leaf nodes 15,Légère baisse de performance par restriction d...,Test de restriction de la complexité des arbres,16:22:07
6,Modele lineaire logistique,Performance inférieure car le modèle linéaire ...,Test d'une famille de modèle linéaire différente,16:22:08
7,feature ecart_rapport_acte,gain proche de 0 (dans le bruit),les deux colonnes sont presque identiques : l'...,16:24:48


sans la feature 82.51 %  |  avec 82.45 %  |  gain -0.24 barre(s)
par predict() : 82.45 %  (doit être égal à 82.45 %)
table à la main : 82.44 %  |  écart -0.06 barre(s)
  ok        la feature s inspire d un constat de la question 3
  ok        votre feature a enrichi la table
  ok        le gain est chiffré en barres
  ok        l écart du contournement est chiffré en barres
  ok        l étape oubliée est nommée


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [34]:

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
# TODO : l'écart de remplissage entre train et test, colonne par colonne.
communes = [c for c in train.columns if c in test.columns and c != CIBLE]
rempl_train = 100 * train[communes].notna().mean()
rempl_test = 100 * test[communes].notna().mean()
ecarts = (rempl_train - rempl_test).sort_values(key=abs, ascending=False)   # tri par écart absolu
display(pd.DataFrame({"rempli en train (%)": rempl_train, "rempli en test (%)": rempl_test,
                      "écart (pts)": rempl_train - rempl_test}).loc[ecarts.index[:5]].round(1))

colonne_trouvee = ecarts.index[0]
ecart_remplissage = float(abs(ecarts.iloc[0]))
seconde = ecarts.index[1]
ecart_seconde = float(abs(ecarts.iloc[1]))
rempl_test_seconde = float(rempl_test[seconde])

pourquoi_pas_la_derivee = (
    f"`{seconde}` arrive deuxième, mais son écart de remplissage n'est que de {ecart_seconde:.1f} point : "
    f"c'est du bruit d'échantillonnage, pas une disparition ni même une dérive. Elle reste remplie à "
    f"{rempl_test_seconde:.0f} % dans le test. Elle n'est donc pas la colonne qui disparaît ; "
    "elle est à écarter pour une autre raison (elle n'existe qu'après l'issue)."
)

remplissage_du_test = float(rempl_test[colonne_trouvee])
taux_trous = 1 - remplissage_du_test / 100

idx = mon_train.index
ym = y.loc[idx]
col = colonne_trouvee

Xa = X_sans_features.loc[idx].copy()          # la table nue : pas de feature de la Q10
Xa[col] = (mon_train[col] if pd.api.types.is_numeric_dtype(mon_train[col])
           else mon_train[col].astype("category"))

Xtr_m, Xval_m, ytr_m, yval_m = train_test_split(Xa, ym, test_size=0.2, stratify=ym, random_state=0)

m_avec = HistGradientBoostingClassifier(random_state=0).fit(Xtr_m, ytr_m)
m_sans = HistGradientBoostingClassifier(random_state=0).fit(Xtr_m.drop(columns=[col]), ytr_m)
score_avec = 100 * m_avec.score(Xval_m, yval_m)
score_sans = 100 * m_sans.score(Xval_m.drop(columns=[col]), yval_m)


s = score_avec / 100
barre_locale = 100 * (s * (1 - s) / len(yval_m)) ** 0.5
gain_interne = score_avec - score_sans                      # en points
gain_en_barres = gain_interne / barre_locale

# TODO : la simulation — la colonne manquante au taux du test, dans la validation.
rng = np.random.default_rng(0)
masque = rng.random(len(Xval_m)) < taux_trous
Xval_sim = Xval_m.copy()
Xval_sim.loc[masque, col] = np.nan
score_simule = 100 * m_avec.score(Xval_sim, yval_m)         # le modèle entraîné AVEC la colonne
chute_en_barres = (score_simule - score_avec) / barre_locale

print(f"colonne trouvée : {colonne_trouvee}  (écart {ecart_remplissage:.1f} pts, remplie à {remplissage_du_test:.1f} % dans le test)")
print(f"validation de {len(yval_m)} lignes → barre de bruit : {barre_locale:.2f} point(s)")
print(f"avec la colonne {score_avec:.2f} %  |  sans {score_sans:.2f} %  |  gain {gain_interne:+.2f} pt = {gain_en_barres:+.2f} barre(s)")
print(f"simulation ({100 * taux_trous:.1f} % de lignes sans la colonne) : {score_simule:.2f} %  "
      f"|  chute {chute_en_barres:+.2f} barre(s)  |  reste vs « sans » : {score_simule - score_sans:+.2f} pt")

decision_A_ECARTER = (
    f"Était-ce la bonne décision ? Oui : la colonne `{colonne_trouvee}` donne {gain_en_barres:+.2f} barre(s) "
    f"de gain en interne, mais quand on la rend manquante comme dans le test, le score tombe à "
    f"{score_simule:.2f} % (chute de {chute_en_barres:+.2f} barre(s)) : le gain interne était un mirage. "
    f"Était-ce pour la bonne raison ? Pas à cause d'un manque d'utilité, puisqu'elle aide en interne : "
    f"la vraie raison est qu'elle n'est renseignée qu'à {remplissage_du_test:.1f} % dans le test, "
    "donc le modèle apprendrait avec une information qu'il n'aura plus en production."
)
controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


,rempli en train (%),rempli en test (%),écart (pts)
canal_prise,100.0,0.5,99.5
motif_absence,18.3,19.0,-0.8
rappel_envoye,33.7,34.1,-0.3
constantes_relevees,57.6,57.3,0.3
solde_credit,94.5,94.5,-0.0


colonne trouvée : canal_prise  (écart 99.5 pts, remplie à 0.5 % dans le test)
validation de 17988 lignes → barre de bruit : 0.26 point(s)
avec la colonne 85.53 %  |  sans 82.55 %  |  gain +2.99 pt = +11.38 barre(s)
simulation (99.5 % de lignes sans la colonne) : 82.45 %  |  chute -11.74 barre(s)  |  reste vs « sans » : -0.09 pt
  ok        l écart de remplissage est un pourcentage
  ok        la colonne nommée est bien celle de VOTRE tri
  ok        la colonne qui dérive est écartée, avec sa raison
  ok        la part du test encore renseignée est un pourcentage
  ok        le gain interne et sa simulation sont chiffrés
  ok        la simulation a été mesurée sur un score à part
  ok        la décision est argumentée, sur les deux questions


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [35]:

import numpy as np

# ── L'outil : les DEUX signes + un signe partiel utile (le vide « presque » assez fort)
def suspects(df, cible, ignorer=(ID, "dossier_id", "patient_id"),
             seuil_vide=50, seuil_vide_partiel=30, seuil_valeur=50):
    """Une ligne par colonne suspecte : nom, écart de remplissage, signe, détail.
       Signe 1 (VIDE)   : écart de remplissage > seuil_vide points entre les deux groupes de la cible.
       Signe 2 (VALEUR) : une valeur qui pèse plus de seuil_valeur points de plus dans un groupe que dans l'autre.
       La cible ne sert QU'À séparer les deux groupes : elle n'est jamais une colonne candidate."""
    classes = sorted(df[cible].dropna().unique())
    g0, g1 = df[df[cible] == classes[0]], df[df[cible] == classes[-1]]
    lignes = []
    for c in df.columns:
        if c == cible or c in ignorer:
            continue
        # signe 1 : le remplissage, groupe par groupe
        ecart = 100 * (g1[c].notna().mean() - g0[c].notna().mean())
        signes, details = [], []
        if abs(ecart) > seuil_vide:
            signes.append("VIDE")
            details.append(f"remplie à {100 * g1[c].notna().mean():.0f} % dans le groupe {classes[-1]} "
                           f"contre {100 * g0[c].notna().mean():.0f} % dans le groupe {classes[0]}")
        elif abs(ecart) > seuil_vide_partiel:
            signes.append("VIDE (partiel)")
            details.append(f"remplie à {100 * g1[c].notna().mean():.0f} % contre {100 * g0[c].notna().mean():.0f} %")
        # signe 2 : la valeur qui ne vit que (presque) dans un groupe — parts calculées sur TOUT le groupe
        part1 = g1[c].value_counts() / len(g1)
        part0 = g0[c].value_counts() / len(g0)
        diff = part1.sub(part0, fill_value=0)
        if len(diff):
            v = diff.abs().idxmax()
            if abs(diff[v]) * 100 > seuil_valeur:
                signes.append("VALEUR")
                details.append(f"la valeur {v!r} pèse {100 * part1.get(v, 0):.1f} % du groupe {classes[-1]} "
                               f"contre {100 * part0.get(v, 0):.1f} % du groupe {classes[0]}")
        if signes:
            lignes.append({"colonne": c, "ecart_remplissage (pts)": round(ecart, 1),
                           "signe": " + ".join(signes), "detail": " ; ".join(details)})
    return pd.DataFrame(lignes, columns=["colonne", "ecart_remplissage (pts)", "signe", "detail"])
# TODO : l'outil appliqué à `train`.
table_suspects = suspects(train, CIBLE)
display(table_suspects)

# ── Le classement : une ligne par colonne signalée. VERDICTS À RELIRE ET À REFORMULER.
jugements = {
    "motif_absence": ("fuite",
        "Un motif d'absence n'existe que si le patient a été absent : il est rempli pour tous les absents et "
        "vide pour tous les présents. Il décrit l'issue au lieu de la prédire, donc il n'existe pas au moment de décider."),
    "duree_consultation": ("fuite",
        "Un patient absent n'a pas de consultation, donc pas de durée : le système écrit -1 à la place. "
        "La valeur -1 veut dire « le patient n'est pas venu », une information connue seulement après l'issue."),
    "constantes_relevees": ("à discuter",
        "Des constantes sont relevées quand le patient est là : un absent n'en a pas, donc le trou suit la réponse. "
        "Mais la colonne n'est pas remplie après coup au sens strict ; à trancher selon le moment où on les relève."),
}
lignes = []
for _, r in table_suspects.iterrows():
    verdict, justif = jugements.get(r["colonne"], (
        "à discuter",
        f"Signalée ({r['signe']}) : {r['detail']}. À qualifier : cette information est-elle connue avant le rendez-vous ?"))
    lignes.append({"colonne": r["colonne"], "verdict": verdict, "justification": justif})
classement_suspects = pd.DataFrame(lignes)
display(classement_suspects)
# TODO : le troisième signe — la sentinelle.
deja = set(table_suspects["colonne"])
candidats = []
for c in train.select_dtypes("number").columns:
    if c in (ID, CIBLE) or c in deja:
        continue
    mn = train[c].min()
    n = int((train[c] == mn).sum())
    if mn < 0 and 10 <= n < 5000:             # une valeur négative REPRISE sur plusieurs lignes
        candidats.append({"colonne": c, "valeur": mn, "lignes": n,
                          "taux d'absence (%)": round(100 * train.loc[train[c] == mn, CIBLE].mean(), 2)})
print("valeurs négatives reprises par plusieurs lignes (à toi de dire lesquelles sont IMPOSSIBLES) :")
display(pd.DataFrame(candidats))

col_s, val_s = "age", -1                       # un âge de -1 n'existe pas
masque = train[col_s] == val_s
sentinelle_trouvee = f"{col_s} = {val_s}"
lignes_porteuses = int(masque.sum())
taux_sur_ces_lignes = float(100 * train.loc[masque, CIBLE].mean())
taux_global = 100 * train[CIBLE].mean()
barre_s = 100 * ((taux_global / 100) * (1 - taux_global / 100) / lignes_porteuses) ** 0.5
print(f"{sentinelle_trouvee} : {lignes_porteuses} lignes | taux d'absence {taux_sur_ces_lignes:.1f} % "
      f"contre {taux_global:.1f} % au global (barre de bruit sur {lignes_porteuses} lignes : {barre_s:.1f} pts)")

proche = abs(taux_sur_ces_lignes - taux_global) < 2 * barre_s
verdict_sentinelle = (
    f"Sur les {lignes_porteuses} lignes où {col_s} vaut {val_s}, le taux d'absence est de "
    f"{taux_sur_ces_lignes:.1f} % contre {taux_global:.1f} % au global, "
    + ("un écart de moins de deux barres : ce n'est pas une différence, donc le code ne connaît pas la réponse. "
       "C'est un défaut de saisie (un code mis à la place d'un âge manquant), à marquer comme manquant."
       if proche else
       "un écart de plus de deux barres : le code semble lié à la réponse, à examiner comme une possible fuite.")
)
controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


,colonne,ecart_remplissage (pts),signe,detail
0,duree_consultation,0.0,VALEUR,la valeur np.int64(-1) pèse 100.0 % du groupe ...
1,constantes_relevees,-39.5,VIDE (partiel),remplie à 25 % contre 65 %
2,motif_absence,100.0,VIDE,remplie à 100 % dans le groupe 1 contre 0 % da...


,colonne,verdict,justification
0,duree_consultation,fuite,"Un patient absent n'a pas de consultation, don..."
1,constantes_relevees,à discuter,Des constantes sont relevées quand le patient ...
2,motif_absence,fuite,Un motif d'absence n'existe que si le patient ...


valeurs négatives reprises par plusieurs lignes (à toi de dire lesquelles sont IMPOSSIBLES) :


,colonne,valeur,lignes,taux d'absence (%)
0,age,-1,46,10.87
1,age_mois,-12,46,10.87


age = -1 : 46 lignes | taux d'absence 10.9 % contre 18.3 % au global (barre de bruit sur 46 lignes : 5.7 pts)
  ok        `suspects` rend une table, pas une liste de mots
  ok        l identifiant et la cible sont exclus
  ok        le signe du VIDE apparaît au moins une fois
  ok        le signe de la VALEUR apparaît au moins une fois
  ok        chaque verdict a été choisi, et le tri n est pas uniforme
  ok        chaque colonne signalée est classée, et chaque verdict argumenté
  ok        la sentinelle est nommée, chiffrée, et jugée


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [36]:
# Écris TA prédiction avec tes mots. Ce texte est un brouillon.
annonce = (
    "Je prédis que, à délai égal, le rappel n'a pas d'effet net (écart de moins de deux barres) : "
    "l'écart global viendrait du délai, car on envoie un rappel quand le rendez-vous est lointain, "
    "et un rendez-vous lointain s'oublie davantage. Je changerais d'avis si, dans la plupart des "
    "tranches de délai, l'écart restait au-dessus de deux barres et de même signe."
)
print(annonce)

Je prédis que, à délai égal, le rappel n'a pas d'effet net (écart de moins de deux barres) : l'écart global viendrait du délai, car on envoie un rappel quand le rendez-vous est lointain, et un rendez-vous lointain s'oublie davantage. Je changerais d'avis si, dans la plupart des tranches de délai, l'écart restait au-dessus de deux barres et de même signe.


In [37]:
# TODO : votre annonce, AVANT de regarder les tranches.

import numpy as np

# ── La colonne « rappel » : trouvée par son nom, à vérifier
candidats = [c for c in mon_train.columns if "rappel" in c.lower()]
print("colonnes contenant « rappel » :", candidats)
assert candidats, "aucune colonne ne contient « rappel » : écris son nom à la main dans COL_RAPPEL"
COL_RAPPEL = candidats[0]
print(mon_train[COL_RAPPEL].value_counts(dropna=False).head())   # vérifie ce que valent les deux groupes

r = mon_train[COL_RAPPEL]
avec_rappel = mon_train["rappel_envoye"].notna()        # « envoyé » = rappel ; vide = pas de rappel
print(int(avec_rappel.sum()), "lignes avec rappel sur", len(mon_train))


taux_avec = 100 * mon_train.loc[avec_rappel, CIBLE].mean()
taux_sans = 100 * mon_train.loc[~avec_rappel, CIBLE].mean()
ecart_global = float(taux_avec - taux_sans)
print(f"absence avec rappel {taux_avec:.2f} %  |  sans {taux_sans:.2f} %  |  écart global {ecart_global:+.2f} point(s)")
# TODO : l'écart global, puis les tranches AVEC leur barre de bruit.
tranche = pd.cut(mon_train["delai_jours"], [-1, 0, 3, 7, 14, 30, 10_000],
                 labels=["0 j", "1-3 j", "4-7 j", "8-14 j", "15-30 j", "> 30 j"])
tranches, tranches_exclues = [], []
for nom, g in mon_train.groupby(tranche, observed=True):
    a = avec_rappel.loc[g.index]
    n1, n0 = int(a.sum()), int((~a).sum())
    if n1 == 0 or n0 == 0:
        tranches_exclues.append({"tranche": str(nom), "avec": n1, "sans": n0,
                                 "raison": "un seul des deux groupes : rien à comparer"})
        continue
    if min(n1, n0) < 500:
        tranches_exclues.append({"tranche": str(nom), "avec": n1, "sans": n0,
                                 "raison": "un des deux groupes a moins de 500 lignes : l'écart serait du hasard"})
        continue
    p1, p0 = g.loc[a, CIBLE].mean(), g.loc[~a, CIBLE].mean()
    ecart = 100 * (p1 - p0)
    barre = 100 * (p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0) ** 0.5        # barre de bruit de l'écart
    tranches.append({"tranche": str(nom), "taux_avec": round(100 * p1, 2), "taux_sans": round(100 * p0, 2),
                     "lignes": min(n1, n0), "ecart": round(ecart, 2), "barre": round(barre, 2),
                     "ecart_en_barres": round(ecart / barre, 2)})
display(pd.DataFrame(tranches))
print("tranches exclues :")
display(pd.DataFrame(tranches_exclues))

# ── 4. La variable qui explique
delai_median_avec = float(mon_train.loc[avec_rappel, "delai_jours"].median())
delai_median_sans = float(mon_train.loc[~avec_rappel, "delai_jours"].median())
nb_nettes = sum(1 for t in tranches if abs(t["ecart_en_barres"]) >= 2)
print(f"délai médian : {delai_median_avec:.0f} j avec rappel  |  {delai_median_sans:.0f} j sans  |  "
      f"tranches avec écart ≥ 2 barres : {nb_nettes} sur {len(tranches)}")
ecarts_t = ", ".join(f"{t['ecart']:+.1f}" for t in tranches)
variable_qui_explique = (
    f"delai_jours : le délai médian est de {delai_median_avec:.0f} jours avec rappel contre "
    f"{delai_median_sans:.0f} jour sans, car le rappel est envoyé surtout aux rendez-vous lointains. "
    f"Globalement l'absence est de {taux_avec:.2f} % avec rappel contre {taux_sans:.2f} % sans "
    f"(écart {ecart_global:+.2f} points), mais à délai égal l'écart change de signe dans les "
    f"{len(tranches)} tranches comparables ({ecarts_t} points) ; un seul dépasse deux barres. "
    "Le délai explique donc l'écart global, mais l'effet propre du rappel reste incertain."
)
# ── 5. L'expérience (brouillon à reformuler avec tes mots)
experience_qui_trancherait = (
    "On tirerait au hasard, parmi des rendez-vous de même nature, ceux qui reçoivent un rappel et ceux qui n'en "
    "reçoivent pas, sans regarder le délai. On comparerait ensuite le taux d'absence des deux groupes. "
    "La comparaison serait honnête parce que le tirage au sort répartit le délai, la clinique et tout le reste "
    "de la même façon dans les deux groupes : la seule différence systématique est alors le rappel."
)
controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


colonnes contenant « rappel » : ['rappel_envoye']
rappel_envoye
NaN       59554
envoyé    30382
Name: count, dtype: int64
30382 lignes avec rappel sur 89936
absence avec rappel 28.35 %  |  sans 13.11 %  |  écart global +15.24 point(s)


,tranche,taux_avec,taux_sans,lignes,ecart,barre,ecart_en_barres
0,4-7 j,24.90,26.43,927,-1.53,1.51,-1.02
1,8-14 j,29.48,32.84,673,-3.36,1.88,-1.79
2,15-30 j,30.85,35.58,607,-4.74,2.02,-2.34


tranches exclues :


,tranche,avec,sans,raison
0,0 j,0,35796,un seul des deux groupes : rien à comparer
1,1-3 j,0,21185,un seul des deux groupes : rien à comparer
2,> 30 j,4445,366,un des deux groupes a moins de 500 lignes : l'...


délai médian : 11 j avec rappel  |  0 j sans  |  tranches avec écart ≥ 2 barres : 1 sur 3
  ok        l annonce est écrite avant les tranches
  ok        l écart global est chiffré, en points
  ok        les tranches portent leur barre, pas seulement leur écart
  ok        les tranches trop fines de VOTRE échantillon sont mises à part
  ok        chaque tranche comparée porte son écart et son effectif
  ok        les effectifs sont comptés
  ok        la variable qui explique est nommée, et les deux médianes sont chiffrées
  ok        l expérience qui trancherait est décrite en trois lignes


### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [38]:
# Le critère de verdict est annoncé ICI, avant la mesure : gain >= 2 barres => confirmée, sinon contredite.
hypothese("delai_x_absences_passees",
          "un rendez-vous lointain pour un patient déjà souvent absent s'oublie encore plus : "
          "signe + (plus d'absences quand le délai ET l'historique d'absences sont élevés tous les deux)",
          "ajouter delai_jours * taux_absence_passe à la table nue, même découpage, gain en barres par rapport à score_interne",
          "contredite si le gain est inférieur à +2 barres (soit moins de 0,44 point) : le modèle capterait déjà "
          "cette interaction avec les deux colonnes séparées")
hypothese("solde_negatif",
          "un solde de crédit négatif (compte débiteur) signale un patient en difficulté : "
          "signe + (plus d'absences)",
          "ajouter (solde_credit < 0) à la table nue, gain en barres, et taux d'absence des lignes concernées",
          "contredite si le gain est inférieur à +2 barres, ou si le taux d'absence des soldes négatifs "
          "reste à moins de deux barres du taux global")
hypothese("solde_manquant",
          "un solde non renseigné (environ 5,5 % de trous) peut signaler un dossier incomplet : "
          "signe + (plus d'absences)",
          "ajouter solde_credit.isna() à la table nue, gain en barres par rapport à score_interne",
          "contredite si le gain est inférieur à +2 barres : le modèle voit déjà les trous de la colonne, "
          "il n'a pas besoin d'un indicateur à part")

# La prédiction de l'essai testé, sous le MÊME nom que l'hypothèse (adapte-la avec tes mots)
prevoir("delai_x_absences_passees",
        "gain dans le bruit (moins de 2 barres)",
        "un ensemble d'arbres combine déjà delai_jours et taux_absence_passe par ses coupes successives : "
        "le produit n'ajoute presque rien")

,feature,ce qu'elle raconte,test,contredite si
0,delai_x_absences_passees,un rendez-vous lointain pour un patient déjà s...,ajouter delai_jours * taux_absence_passe à la ...,contredite si le gain est inférieur à +2 barre...


,feature,ce qu'elle raconte,test,contredite si
0,delai_x_absences_passees,un rendez-vous lointain pour un patient déjà s...,ajouter delai_jours * taux_absence_passe à la ...,contredite si le gain est inférieur à +2 barre...
1,solde_negatif,un solde de crédit négatif (compte débiteur) s...,"ajouter (solde_credit < 0) à la table nue, gai...",contredite si le gain est inférieur à +2 barre...


,feature,ce qu'elle raconte,test,contredite si
0,delai_x_absences_passees,un rendez-vous lointain pour un patient déjà s...,ajouter delai_jours * taux_absence_passe à la ...,contredite si le gain est inférieur à +2 barre...
1,solde_negatif,un solde de crédit négatif (compte débiteur) s...,"ajouter (solde_credit < 0) à la table nue, gai...",contredite si le gain est inférieur à +2 barre...
2,solde_manquant,"un solde non renseigné (environ 5,5 % de trous...","ajouter solde_credit.isna() à la table nue, ga...",contredite si le gain est inférieur à +2 barre...


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,16:21:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",16:21:26
2,max_depth=3 sur le boosting,pire que score_interne,des arbres peu profonds captent moins d'intera...,16:21:47
3,forêt aléatoire,un peu moins bon que le boosting,"la forêt moyenne des arbres indépendants, alor...",16:21:47
4,Imputation âge par médiane,Gain neutre car l'algorithme gère les valeurs ...,Test d'imputation des valeurs manquantes,16:22:05
5,Max leaf nodes 15,Légère baisse de performance par restriction d...,Test de restriction de la complexité des arbres,16:22:07
6,Modele lineaire logistique,Performance inférieure car le modèle linéaire ...,Test d'une famille de modèle linéaire différente,16:22:08
7,feature ecart_rapport_acte,gain proche de 0 (dans le bruit),les deux colonnes sont presque identiques : l'...,16:24:48
8,delai_x_absences_passees,gain dans le bruit (moins de 2 barres),un ensemble d'arbres combine déjà delai_jours ...,16:24:54


In [39]:
# TODO : trois hypothèses, ici.
# hypothese("ma feature", "ce qu'elle raconte, avec le signe", "comment je la teste",
#           "ce qui la contredirait")
def score_avec_colonne(nom, serie, graine=0):
    """Le modèle de la mise en place + UNE colonne de plus, même découpage que score_interne."""
    Xf = X_sans_features.copy()
    # TODO : une hypothèse TESTÉE, sous son propre nom.
    Xf[nom] = serie
    a, b, c, d = train_test_split(Xf, y, test_size=0.2, stratify=y, random_state=graine)
    return HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)   

feature_test = X_sans_features["delai_jours"] * X_sans_features["taux_absence_passe"]

# essai("ma feature", lambda: modele.score(Xval, yval))
score_h = essai("delai_x_absences_passees",
                lambda: score_avec_colonne("delai_x_absences_passees", feature_test),
                changement="ajout de delai_jours * taux_absence_passe")

gain_h = (score_h - score_interne) / erreur_type
confirmee_ou_contredite = "confirmée" if gain_h >= 2 else "contredite"      # le critère annoncé plus haut
print(f"score {score_h:.2f} %  |  score_interne {score_interne:.2f} %  |  gain {gain_h:+.2f} barre(s)  →  {confirmee_ou_contredite}")

suite_donnee = (
    f"Le gain mesuré est de {gain_h:+.2f} barre(s) : l'hypothèse est {confirmee_ou_contredite}, selon le critère "
    "annoncé avant la mesure (au moins +2 barres). "
    + ("Je garde la feature et je la vérifie sur une autre graine avant de la retenir."
       if gain_h >= 2 else
       "Je ne la garde pas : ce que cela m'apprend, c'est que le modèle exploite déjà cette interaction à partir "
       "des deux colonnes séparées. Je teste ensuite une hypothèse d'une autre nature (solde_negatif ou solde_manquant).")
)

controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"
2,max_depth=3 sur le boosting,max_depth=3 (réglage),pire que score_interne,82.49,"oui, à 16:21:47"
3,forêt aléatoire,"RandomForest 100 arbres, profondeur 12 (famill...",un peu moins bon que le boosting,81.84,"oui, à 16:21:47"
4,Imputation âge par médiane,voir la question,Gain neutre car l'algorithme gère les valeurs ...,8244.10,"oui, à 16:22:05"
5,Max leaf nodes 15,voir la question,Légère baisse de performance par restriction d...,8249.44,"oui, à 16:22:07"
6,Modele lineaire logistique,voir la question,Performance inférieure car le modèle linéaire ...,8209.08,"oui, à 16:22:08"
7,delai_x_absences_passees,ajout de delai_jours * taux_absence_passe,gain dans le bruit (moins de 2 barres),82.45,"oui, à 16:24:54"


score 82.45 %  |  score_interne 82.51 %  |  gain -0.24 barre(s)  →  contredite
  ok        trois hypothèses
  ok        chacune dit ce qui la contredirait, précisément
  ok        une hypothèse a été testée sous son propre nom
  ok        le verdict est confirmée ou contredite
  ok        la suite donnée est écrite
  ok        les trois tableaux sont remplis avant le rapport


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [40]:
# TODO : une famille par colonne, du vocabulaire fermé ci-dessus.
COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal', 'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition', 'utilisable']

familles = {
    'id_client':     'identifiant',
    'date_defaut':   "connue après l'issue",
    'statut_6_mois': "connue après l'issue",
    'montant':       'utilisable',
    'canal':         'utilisable',
    'score_interne': 'utilisable',
}
# TODO : une phrase de justification par colonne.
justifications = {
    'id_client': "Il désigne un client au lieu de le décrire : chaque valeur est propre à une ligne, "
                 "donc il ne peut rien apprendre à un modèle sur un nouveau client.",
    'date_defaut': "Elle n'est renseignée que si l'emprunteur a fait défaut : sa présence ou son absence "
                   "donne la réponse, et elle n'existe pas encore au moment d'accorder le crédit.",
    'statut_6_mois': "Il est relevé six mois après l'octroi : au moment de décider, on ne sait pas encore "
                     "si le client sera à jour ou en retard, donc il connaît déjà l'issue.",
    'montant': "Le montant est demandé par le client au moment de la demande : il est connu avant la "
               "décision et il décrit le dossier.",
    'canal': "Le canal (agence, web, téléphone) est connu dès le dépôt de la demande et décrit la ligne ; "
             "rien dans l'en-tête n'indique qu'il cesserait d'être alimenté.",
    'score_interne': "Il est calculé au moment de la demande, donc disponible quand on décide, "
                     "et il décrit le risque du dossier.",
}

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


  ok        les six colonnes sont classées
  ok        le vocabulaire est celui des quatre familles
  ok        la colonne du défaut n est pas classée « utilisable »
  ok        l identifiant est reconnu comme tel
  ok        le score calculé à la demande reste utilisable
  ok        les six justifications sont écrites, une phrase chacune


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [41]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


CONSTATS CHIFFRÉS 



,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune ligne dupliquée,0 ligne dupliquée,sur les 89936 lignes de mon_train (hors identi...,garder les lignes
4,duree_consultation vaut -1 chez la quasi-total...,16420 absents sur 16420 portent -1,contre 91 présents sur 73516 portant -1,écarter la colonne car elle est connue après l...


HYPOTHÈSES DE FEATURES 



,feature,ce qu'elle raconte,test,contredite si
0,delai_x_absences_passees,un rendez-vous lointain pour un patient déjà s...,ajouter delai_jours * taux_absence_passe à la ...,contredite si le gain est inférieur à +2 barre...
1,solde_negatif,un solde de crédit négatif (compte débiteur) s...,"ajouter (solde_credit < 0) à la table nue, gai...",contredite si le gain est inférieur à +2 barre...
2,solde_manquant,"un solde non renseigné (environ 5,5 % de trous...","ajouter solde_credit.isna() à la table nue, ga...",contredite si le gain est inférieur à +2 barre...


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 16:21:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 16:21:26"
2,max_depth=3 sur le boosting,max_depth=3 (réglage),pire que score_interne,82.49,"oui, à 16:21:47"
3,forêt aléatoire,"RandomForest 100 arbres, profondeur 12 (famill...",un peu moins bon que le boosting,81.84,"oui, à 16:21:47"
4,Imputation âge par médiane,voir la question,Gain neutre car l'algorithme gère les valeurs ...,8244.10,"oui, à 16:22:05"
5,Max leaf nodes 15,voir la question,Légère baisse de performance par restriction d...,8249.44,"oui, à 16:22:07"
6,Modele lineaire logistique,voir la question,Performance inférieure car le modèle linéaire ...,8209.08,"oui, à 16:22:08"
7,delai_x_absences_passees,ajout de delai_jours * taux_absence_passe,gain dans le bruit (moins de 2 barres),82.45,"oui, à 16:24:54"


VOS CONTRÔLES 



,contrôle,tient
0,"la part d absents est un pourcentage, et c est...",True
1,"`ma_reference` est un pourcentage, au-dessus d...",True
2,les deux chiffres sont complémentaires,True
3,le prédicteur ne répond que des 0 et des 1,True
4,son score COÏNCIDE avec votre référence,True
...,...,...
95,le vocabulaire est celui des quatre familles,True
96,la colonne du défaut n est pas classée « utili...,True
97,l identifiant est reconnu comme tel,True
98,le score calculé à la demande reste utilisable,True


LA RELECTURE 

  ok        votre identifiant GitHub est renseigné
  ok        cinq constats
  ok        au moins deux constats mesurés par vous
  ok        chaque constat porte un chiffre ET une comparaison
  ok        trois hypothèses
  ok        chaque hypothèse dit ce qui la contredirait
  ok        au moins trois essais à vous — les amorces ne comptent pas
  ok        chaque essai porte un attendu — un signe ET une raison
  ok        les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle
  ok        une hypothèse a été testée sous son propre nom


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
